PAT: Port Address Translation is a technique used in networking to allow multiple devices on a local network to share a single public IP address when accessing the internet. It modifies the source port numbers of outgoing packets and keeps track of these translations in a table, enabling the return traffic to be correctly routed back to the originating device. This is commonly used in home and small business networks to conserve public IP addresses and enhance security.

In [31]:
import random

router_ip = "168.29.72.1"
router_port = -1
PAT_address_book = {}
occupied_router_ports = []

class Packet:
    def __init__(self):
        self.source_ip = generate_ip("private")
        self.source_port = generate_port()
        self.destination_ip = generate_ip("public")
        self.destination_port = generate_port()
        #print("packet created")
    
    def content(self):
        print(f"Source IP: {self.source_ip}, Source Port: {self.source_port}," 
              f" Destination IP: {self.destination_ip}, Destination Port: {self.destination_port}")

    def update_source(self, new_ip, new_port):
        self.source_ip = new_ip
        self.source_port = new_port

    def update_destination(self, new_ip, new_port):
        self.destination_ip = new_ip
        self.destination_port = new_port

def generate_ip(ip_type):

    if ip_type == "private":
        private_ip = "192.168.1." + str(random.randint(1, 10))
        return private_ip

    elif ip_type == "public":
        public_ip = ""
        for counter in range(4):
            public_ip += str(random.randint(193,205))
            if counter < 3:
                public_ip += "."
            #public_ip += "."
        return public_ip

def generate_port():
    port_number = str(random.randint(0, 20))
    return port_number

def reverse_content(packet_data):
    temp = packet_data.source_ip
    packet_data.source_ip = packet_data.destination_ip
    packet_data.destination_ip = temp

    temp = packet_data.source_port
    packet_data.source_port = packet_data.destination_port
    packet_data.destination_port = temp

def address_translation(packet_data, traffic_type):
    global router_port
    global PAT_address_book
    global occupied_router_ports

    if traffic_type == "outbound":
        if router_port >= 65535:
            print("No more ports available for translation.")
            return

        router_port += 1
        PAT_address_book[router_port] = (packet_data.source_ip, packet_data.source_port)
        occupied_router_ports.append(router_port)
        packet_data.update_source(router_ip, router_port)
        return packet_data

    elif traffic_type == "inbound":
        if packet_data.destination_port in PAT_address_book.keys():
            private_source_ip, private_source_port = PAT_address_book[packet_data.destination_port]
            del PAT_address_book[packet_data.destination_port] # freeing the port after translation
            packet_data.update_destination(private_source_ip, private_source_port)
            
            return packet_data
        else:
            print("No translation found for the given destination port.")
            return packet_data

def main():
    captured_packet = []
    translated_packet = []

    #abc = Packet()
    #abc.content()

    print("Outbound packets")

    for i in range(5):
        captured_packet.append(Packet())
        captured_packet[i].content() # Prints the original packet details

    print("\nAfter PAT")

    for i in range(5):
        # packets are now created by endpoints and are ready to be sent to the router for translation
        captured_packet[i] = address_translation(captured_packet[i], "outbound") # Translates the packet and updates the source IP and port
        captured_packet[i].content()
        # The translated packet is now ready to be sent to the destination endpoint

    #print(PAT_address_book)

    #now reversing the translation to simulate the return packet from the destination endpoint
    print("\nInbound packets")
    for i in range(5):
        reverse_content(captured_packet[i]) # Reverses the source and destination IP and port
        captured_packet[i].content() # Prints the reversed packet details

    print("\nreverse PAT translation")

    for i in range(5):
        captured_packet[i] = address_translation(captured_packet[i], "inbound") # Translates the packet back to the original source IP and port
        captured_packet[i].content() # Prints the translated packet details

if __name__ == "__main__":
    main()
    

Outbound packets
Source IP: 192.168.1.1, Source Port: 14, Destination IP: 201.198.200.194, Destination Port: 10
Source IP: 192.168.1.10, Source Port: 19, Destination IP: 201.201.205.202, Destination Port: 20
Source IP: 192.168.1.6, Source Port: 1, Destination IP: 198.196.203.204, Destination Port: 18
Source IP: 192.168.1.7, Source Port: 4, Destination IP: 202.195.199.193, Destination Port: 20
Source IP: 192.168.1.1, Source Port: 18, Destination IP: 199.201.194.195, Destination Port: 6

After PAT
Source IP: 168.29.72.1, Source Port: 0, Destination IP: 201.198.200.194, Destination Port: 10
Source IP: 168.29.72.1, Source Port: 1, Destination IP: 201.201.205.202, Destination Port: 20
Source IP: 168.29.72.1, Source Port: 2, Destination IP: 198.196.203.204, Destination Port: 18
Source IP: 168.29.72.1, Source Port: 3, Destination IP: 202.195.199.193, Destination Port: 20
Source IP: 168.29.72.1, Source Port: 4, Destination IP: 199.201.194.195, Destination Port: 6

Inbound packets
Source IP: 20